# Bias–Variance Trade-off
### Компромисс между смещением и разбросом

Это вводный разбор одной из центральных идей машинного обучения — **почему модель ошибается и какими вообще бывают ошибки**. Мы проследим мысль от начала до конца и все потрогаем на интерактивных графиках: будем крутить ползунки и смотреть, что меняется.

> **Аббревиатуры, которые встретятся.** **BVT** (bias–variance trade-off) — компромисс смещения и разброса; **MSE** (mean squared error) — средняя квадратичная ошибка. Все термины по ходу будем расшифровывать.

Сюжет занятия:
1. Истинная зависимость и шум — откуда вообще берётся ошибка.
2. Что такое «сложность модели» (даже у линейной!).
3. Почему модель «пляшет»: смещение и разброс.
4. Формула, которая всё связывает.
5. Дерево против линейной модели.
6. Как обмануть компромисс — усреднение (мостик к ансамблям).

## 0. Подготовка

Сначала создадим **демонстрационные данные** — так мы сможем сами задать «истину» и потом проверять, насколько модель к ней приблизилась (в реальной жизни истину мы не знаем). Эта ячейка задаёт истинную зависимость $f(x)$ и функцию, которая порождает зашумлённую выборку.

In [48]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor

# The TRUE dependency f(x). In real life it is unknown; here we set it ourselves.
def f_true(x):
    return np.sin(2 * np.pi * x)

X_GRID = np.linspace(0, 1, 200)   # dense grid used to draw smooth curves

def make_dataset(n=30, noise=0.25, seed=None):
    """Sample a noisy training set: y = f(x) + gaussian noise."""
    r = np.random.default_rng(seed)
    x = np.sort(r.uniform(0, 1, n))
    y = f_true(x) + r.normal(0, noise, n)
    return x, y

def fit_predict(x, y, family, complexity):
    """Fit a model of the given family/complexity and predict on X_GRID."""
    xr = x.reshape(-1, 1)
    if family == 'линейная':
        model = LinearRegression()                                  # always a straight line
    elif family == 'полином':
        model = make_pipeline(PolynomialFeatures(int(complexity)), LinearRegression())
    else:  # 'дерево'
        model = DecisionTreeRegressor(max_depth=int(complexity))
    model.fit(xr, y)
    return model.predict(X_GRID.reshape(-1, 1))

## 1. Истинная зависимость и шум

Представим, что в природе есть **истинная зависимость** между признаком `x` и целевой величиной `y`. Обозначим её $f(x)$. Например: «чем выше температура, тем больше продаж мороженого» — где-то существует честная кривая, описывающая эту связь.

Беда в том, что мы **никогда не видим $f(x)$ напрямую**. Мы наблюдаем только данные, в которые подмешан **шум** $\varepsilon$ (греческая буква «эпсилон», от англ. *noise* — шум) — случайная, как правило непредсказуемая добавка:

$$y = f(x) + \varepsilon$$

Где
$$\varepsilon\sim\mathcal N(0,\sigma^2)$$

Откуда берётся шум? Ошибки измерения, влияние факторов, которые мы не учли, просто случайность мира. Ключевое свойство: шум **переменчив**, в среднем равен нулю, но в каждой конкретной точке непредсказуем.

Покрутите уровень шума $\sigma$ (греческая «сигма» — стандартное отклонение шума, то есть мера его силы). Красные отрезки — это и есть шум: промах наблюдения мимо истинной кривой.

In [49]:
# @title  Крутилка: истинная зависимость и шум
def show_noise(noise):
    x, y = make_dataset(n=40, noise=noise, seed=1)
    plt.figure(figsize=(8, 5))
    plt.plot(X_GRID, f_true(X_GRID), 'g-', lw=2.5, label='истинная зависимость f(x)')
    plt.scatter(x, y, color='black', zorder=3, label='наблюдения (с шумом)')
    for xi, yi in zip(x, y):
        plt.plot([xi, xi], [f_true(xi), yi], color='red', alpha=0.4, lw=1)
    plt.title(f'Уровень шума σ = {noise:.2f}')
    plt.xlabel('x'); plt.ylabel('y'); plt.ylim(-2.5, 2.5); plt.legend()
    plt.show()

interact(show_noise, noise=FloatSlider(value=0.25, min=0, max=1, step=0.05, description='шум σ'));

interactive(children=(FloatSlider(value=0.25, description='шум σ', max=1.0, step=0.05), Output()), _dom_classe…

**Вывод этого шага.** Шум нельзя предсказать — это случайность по определению. Значит, какую бы идеальную модель мы ни построили, часть ошибки убрать невозможно. Эта «неустранимая» часть — **шумовой пол** ошибки, равный $\sigma^2$. Запомним: это первая из трёх составляющих ошибки.

## 2. Что такое «сложность модели»

**Сложность модели** (её ещё называют *гибкостью* или *ёмкостью*, англ. *capacity*) — это способность модели гнуться под данные: как много разных форм она может принять.

- У **линейной модели** (прямая линия) сложность минимальна: что бы ни происходило с данными, она остаётся прямой. У неё всего два «рычага» — наклон и сдвиг. Но это **тоже модель со своей (низкой) сложностью** — важно это понимать.
- У **полинома** сложность задаётся его **степенью**: степень 1 — прямая, степень 2 — парабола, степень 9 — кривая, способная извиваться через множество точек.
- У **решающего дерева** роль сложности играет **глубина**: чем глубже дерево, тем на более мелкие кусочки оно режет данные.

Покрутите ползунок сложности и переключайте семейство моделей. Обратите внимание: **линейная модель не реагирует на ползунок** — она всегда прямая (её сложность фиксирована). А полином и дерево с ростом сложности всё сильнее «облизывают» каждую точку, включая шумовые.

In [ ]:
# @title Крутилка: что такое «сложность модели»
def show_complexity(family, complexity):
    x, y = make_dataset(n=100, noise=0.5, seed=3)
    pred = fit_predict(x, y, family, complexity)
    plt.figure(figsize=(8, 5))
    plt.plot(X_GRID, f_true(X_GRID), 'g-', lw=2.5, label='истина f(x)')
    plt.scatter(x, y, color='black', zorder=3, label='обучающие данные')
    label = family if family == 'линейная' else f'{family} (сложность = {complexity})'
    plt.plot(X_GRID, pred, 'b-', lw=2, label=label)
    plt.title('Чем выше сложность, тем сильнее модель гнётся под данные')
    plt.xlabel('x'); plt.ylabel('y'); plt.ylim(-2.5, 2.5); plt.legend()
    plt.show()

interact(show_complexity,
         family=Dropdown(options=['линейная', 'полином', 'дерево'], value='полином', description='семейство'),
         complexity=IntSlider(value=3, min=1, max=30, step=1, description='сложность'));

interactive(children=(Dropdown(description='семейство', index=1, options=('линейная', 'полином', 'дерево'), va…

Когда модель проходит ровно через все точки, включая шумовые, она начинает **запоминать шум** вместо того, чтобы уловить истинную зависимость. Это **переобучение** (англ. *overfitting*) — к нему мы ещё вернёмся.

## 3. Почему модель «пляшет»: повторная выборка

Вот ключевая мысль всего разбора. Наши обучающие данные — это **одна случайная выборка** из мира. Если бы мы собрали данные заново, точки из-за шума легли бы иначе, и обученная модель получилась бы **другой**.

Проведём мысленный эксперимент: соберём данные много раз и каждый раз обучим модель. Получим целое «облако» моделей. Ниже именно оно: бледные синие линии — модели на разных выборках, жирная красная — их **средняя**, зелёная — истина.

Дадим имена двум типам ошибки:

- **Смещение** (англ. *bias*) — на сколько модель промахивается мимо истины **в среднем**. Это систематическая ошибка: если красная (средняя) линия далеко от зелёной (истинной) — смещение велико.
- **Разброс** (англ. *variance*, «вариативность») — на сколько модель **пляшет от выборки к выборке**. Если синие линии разлетаются широко — разброс велик.

Покрутите сложность и понаблюдайте за облаком. Внизу заголовка — посчитанные $\text{bias}^2$ и variance.

In [51]:
# @title Крутилка: повторная выборка — смещение и разброс
def show_spaghetti(family, complexity, n_models=40):
    preds = np.array([fit_predict(*make_dataset(30, 0.25, seed=k), family, complexity)
                      for k in range(n_models)])
    truth = f_true(X_GRID)
    bias2 = np.mean((preds.mean(axis=0) - truth) ** 2)
    var = np.mean(preds.var(axis=0))
    plt.figure(figsize=(8, 5))
    for p in preds:
        plt.plot(X_GRID, p, color='steelblue', alpha=0.12)
    plt.plot(X_GRID, truth, 'g-', lw=2.5, label='истина f(x)')
    plt.plot(X_GRID, preds.mean(axis=0), 'r-', lw=2.5, label='средняя модель')
    title_family = family if family == 'линейная' else f'{family}, сложность = {complexity}'
    plt.title(f'{title_family}   |   bias² = {bias2:.3f},  variance = {var:.3f}')
    plt.xlabel('x'); plt.ylabel('y'); plt.ylim(-2.5, 2.5); plt.legend()
    plt.show()

interact(show_spaghetti,
         family=Dropdown(options=['линейная', 'полином', 'дерево'], value='полином', description='семейство'),
         complexity=IntSlider(value=3, min=1, max=30, step=1, description='сложность'),
         n_models=IntSlider(value=40, min=1, max=70, step=1, description='Количество моделей'));

interactive(children=(Dropdown(description='семейство', index=1, options=('линейная', 'полином', 'дерево'), va…

Вы увидели закономерность:

- **Низкая сложность** (например, прямая): облако тесное (**малый разброс**), но средняя линия систематически мимо изогнутой истины (**большое смещение**). Это **недообучение** (англ. *underfitting*).
- **Высокая сложность**: облако разлетается (**большой разброс**), но в среднем ложится на истину (**малое смещение**). Это дорога к **переобучению**.

Смещение и разброс тянут в разные стороны — в этом и есть **компромисс** (trade-off).

## 4. Разложение ошибки: формула компромисса

Математика подтверждает наблюдение. Ожидаемую квадратичную ошибку (MSE) модели в точке можно разложить ровно на три слагаемых:

$$\underbrace{\mathbb{E}\big[(y - \hat{a}(x))^2\big]}_{\text{полная ошибка}} = \underbrace{\text{bias}^2}_{\text{смещение}^2} + \underbrace{\text{variance}}_{\text{разброс}} + \underbrace{\sigma^2}_{\text{шум}}$$

Здесь $\hat{a}(x)$ (читается «а с крышкой») — прогноз нашей обученной модели, а $\mathbb{E}$ — усреднение по всем возможным обучающим выборкам.

Три источника ошибки:
1. **Смещение²** — насколько в среднем мимо. **Падает** с ростом сложности.
2. **Разброс** — насколько неустойчиво. **Растёт** с ростом сложности.
3. **Шум $\sigma^2$** — неустранимый пол, от модели не зависит.

Покрутите степень полинома. Слева — облако моделей, справа — те же три слагаемых как функции сложности. Видно, как полная ошибка (чёрная) сначала падает (уходит смещение), затем растёт (разрастается разброс), образуя **U-образную кривую**. Дно этой кривой — оптимальная сложность.

In [52]:
# @title Крутилка: компромисс — двигаем сложность, смотрим на ошибку
# Precompute the bias^2 / variance curves across polynomial degrees once (kept snappy).
DEGREES = list(range(1, 11))
NOISE = 0.25

def _bias_var_for_degree(d, n_models=60):
    preds = np.array([fit_predict(*make_dataset(30, NOISE, seed=k), 'полином', d)
                      for k in range(n_models)])
    truth = f_true(X_GRID)
    return np.mean((preds.mean(0) - truth) ** 2), np.mean(preds.var(0))

_BV = np.array([_bias_var_for_degree(d) for d in DEGREES])
_BIAS2, _VAR = _BV[:, 0], _BV[:, 1]
_TOTAL = _BIAS2 + _VAR + NOISE ** 2

def show_tradeoff(degree):
    preds = np.array([fit_predict(*make_dataset(30, NOISE, seed=k), 'полином', degree)
                      for k in range(40)])
    truth = f_true(X_GRID)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for p in preds:
        ax[0].plot(X_GRID, p, color='steelblue', alpha=0.12)
    ax[0].plot(X_GRID, truth, 'g-', lw=2.5, label='истина')
    ax[0].plot(X_GRID, preds.mean(0), 'r-', lw=2.5, label='средняя модель')
    ax[0].set_ylim(-2.5, 2.5); ax[0].legend(); ax[0].set_xlabel('x')
    ax[0].set_title(f'Облако моделей: полином степени {degree}')

    ax[1].plot(DEGREES, _BIAS2, 'o-', label='bias² (смещение)')
    ax[1].plot(DEGREES, _VAR, 'o-', label='variance (разброс)')
    ax[1].plot(DEGREES, _TOTAL, 'o-', color='black', label='полная ошибка')
    ax[1].axvline(degree, color='gray', ls='--')
    ax[1].set_xlabel('сложность (степень полинома)'); ax[1].set_ylabel('ошибка'); ax[1].legend()
    ax[1].set_title('Разложение ошибки на bias² + variance + шум')
    plt.tight_layout()
    plt.show()

interact(show_tradeoff, degree=IntSlider(value=3, min=1, max=10, step=1, description='степень'));

interactive(children=(IntSlider(value=3, description='степень', max=10, min=1), Output()), _dom_classes=('widg…

<details>
<summary><b>Для любознательных:</b> откуда берётся разложение на три части</summary>

<br>

Пусть $y = f(x) + \varepsilon$, где $\mathbb{E}[\varepsilon]=0$, $\mathrm{Var}(\varepsilon)=\sigma^2$, а $\hat{a}(x)$ — прогноз модели, обученной на случайной выборке. Обозначим $\bar{a}(x) = \mathbb{E}[\hat{a}(x)]$ — средний прогноз по всем возможным выборкам. Тогда

$$\mathbb{E}\big[(y-\hat{a})^2\big] = \mathbb{E}\big[(f + \varepsilon - \hat{a})^2\big].$$

Добавим и вычтем $\bar{a}$ внутри скобки: $f + \varepsilon - \hat{a} = (f - \bar{a}) + \varepsilon + (\bar{a} - \hat{a})$, и раскроем квадрат. Все три перекрёстных члена обнуляются, потому что $\mathbb{E}[\varepsilon]=0$ и $\mathbb{E}[\bar{a} - \hat{a}]=0$ (а шум $\varepsilon$ независим от обучения). Остаётся

$$= \underbrace{(f - \bar{a})^2}_{\text{bias}^2} + \underbrace{\mathbb{E}\big[(\hat{a} - \bar{a})^2\big]}_{\text{variance}} + \underbrace{\sigma^2}_{\text{шум}}.$$

То есть смещение — это промах *среднего* прогноза мимо истины, разброс — разлёт прогнозов вокруг своего собственного среднего, а шум вообще не зависит от модели.

</details>

## 5. Дерево против линейной модели: почему они ведут себя по-разному

Компромисс универсален, но разные семейства моделей стоят в разных его точках «по характеру».

- **Линейная модель** делает сильное предположение: «зависимость — прямая». Это жёсткая модель: от выборки к выборке она почти не меняется (**малый разброс**), но если истина изогнута — обречена мазать (**большое смещение**). Заметьте: линейная модель **в принципе не способна** повторить кривую, сколько данных ей ни дай.
- **Глубокое дерево** почти не делает предположений: режет данные на кусочки и подстраивается под каждый. Поэтому в среднем оно может повторить любую форму (**малое смещение**), но сильно пляшет от выборки к выборке (**большой разброс**). Вспомните из занятия про деревья: они дают «ступеньки» и не умеют экстраполировать.

Слева — линейная модель (всегда прямая), справа — дерево с регулируемой глубиной. Сравните **ширину облаков** (разброс) и **положение средней линии** относительно истины (смещение).

In [53]:
# @title Крутилка: дерево против линейной модели
def compare_models(tree_depth):
    truth = f_true(X_GRID)
    lin = np.array([fit_predict(*make_dataset(30, 0.25, seed=k), 'линейная', 1) for k in range(40)])
    tre = np.array([fit_predict(*make_dataset(30, 0.25, seed=k), 'дерево', tree_depth) for k in range(40)])
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    for p in lin:
        ax[0].plot(X_GRID, p, color='steelblue', alpha=0.12)
    ax[0].plot(X_GRID, truth, 'g-', lw=2.5)
    ax[0].plot(X_GRID, lin.mean(0), 'r-', lw=2.5)
    ax[0].set_ylim(-2.5, 2.5); ax[0].set_xlabel('x')
    ax[0].set_title(f'Линейная: bias² = {np.mean((lin.mean(0)-truth)**2):.3f}, '
                    f'var = {np.mean(lin.var(0)):.3f}')

    for p in tre:
        ax[1].plot(X_GRID, p, color='steelblue', alpha=0.12)
    ax[1].plot(X_GRID, truth, 'g-', lw=2.5, label='истина')
    ax[1].plot(X_GRID, tre.mean(0), 'r-', lw=2.5, label='средняя модель')
    ax[1].set_ylim(-2.5, 2.5); ax[1].set_xlabel('x'); ax[1].legend()
    ax[1].set_title(f'Дерево (глубина {tree_depth}): bias² = {np.mean((tre.mean(0)-truth)**2):.3f}, '
                    f'var = {np.mean(tre.var(0)):.3f}')

    plt.tight_layout()
    plt.show()

interact(compare_models, tree_depth=IntSlider(value=5, min=1, max=10, step=1, description='глубина'));

interactive(children=(IntSlider(value=5, description='глубина', max=10, min=1), Output()), _dom_classes=('widg…

Чётко видны два «характера»: линейная — стабильная, но негибкая (высокое смещение, низкий разброс); дерево — гибкое, но капризное (низкое смещение, высокий разброс). Каждая модель — это свой компромисс между этими крайностями.

## 6. Как обмануть компромисс: усреднение

Со смещением борются, подбирая сложность под задачу. Но с разбросом есть отдельный, почти «бесплатный» приём.

Идея: если у нас много моделей, каждая из которых **в среднем правильная (малое смещение), но пляшущая (большой разброс)**, их можно **усреднить**. Среднее множества шумных, но несмещённых прогнозов гораздо устойчивее любого из них по отдельности — случайные «пляски» в разные стороны взаимно гасятся.

Чтобы увидеть это в формуле, нам понадобятся два понятия.

**Что такое дисперсия (Var).** Дисперсия случайной величины $Z$ — это средний квадрат её отклонения от своего же среднего:
$$\mathrm{Var}(Z) = \mathbb{E}\big[(Z - \mathbb{E}[Z])^2\big].$$
Простыми словами — «насколько сильно $Z$ гуляет вокруг своего среднего». У нас $Z = a_i(x)$ — прогноз модели в фиксированной точке $x$, а гуляет он из-за случайности обучающей выборки.

**Что такое $v$ — дисперсия одной модели.** Обозначим $v = \mathrm{Var}\big(a_i(x)\big)$ — разброс прогноза **одной** базовой модели в точке $x$. Это ровно тот самый *variance* (разброс) из разделов 3–4.

> ⚠️ **Не путать с шумом!** В разделе 1 буквой $\sigma^2$ мы обозначали **шум данных** — разброс наблюдений $y$ вокруг истины. Здесь $v$ — это разброс **модели** от выборки к выборке. Это разные величины (хоть обе и измеряются дисперсией): шум живёт в данных, $v$ — в модели.

**Что такое попарная корреляция $\rho$ и откуда она берётся.** Когда мы складываем несколько случайных величин, дисперсия суммы зависит не только от их собственных дисперсий, но и от того, **насколько они «ходят вместе»**. Эту согласованность измеряет коэффициент корреляции $\rho$ (греческая «ро», от 0 до 1):
- $\rho = 0$ — модели независимы, ошибаются каждая по-своему (идеально для усреднения: ошибки гасят друг друга);
- $\rho = 1$ — модели одинаковы, ошибаются синхронно (усреднять бесполезно — среднее равно одной модели).

Откуда вообще берётся это условие? Наши модели обучаются на **похожих** данных (бутстреп-выборки берутся из одной исходной выборки и потому пересекаются), значит, и ошибаются они похоже — их прогнозы **скоррелированы**. Сделать их полностью независимыми не выйдет, поэтому $\rho > 0$ — это неизбежная плата за то, что данные у нас одни на всех.

**Сама формула.** Можно показать (вывод — в блоке «для любознательных» ниже), что дисперсия среднего из $M$ таких моделей равна:
$$\mathrm{Var}\!\Big(\tfrac{1}{M}\sum_{i=1}^{M} a_i\Big) = \underbrace{\rho\, v}_{\text{неустранимый «пол»}} + \underbrace{\frac{1-\rho}{M}\, v}_{\text{тает с ростом } M}.$$

Читаем по частям:
- Второе слагаемое $\frac{1-\rho}{M}\,v$ **уменьшается** с ростом числа моделей $M$ и при $M\to\infty$ обращается в ноль — вот за счёт чего усреднение убивает разброс.
- Первое слагаемое $\rho\, v$ — **пол**, ниже которого простым добавлением моделей не опуститься. Чем сильнее модели похожи ($\rho$ ближе к 1), тем выше пол. Отсюда вывод: модели надо делать **разными**.
- Смещение при усреднении **не меняется** (среднее несмещённых моделей остаётся несмещённым) — поэтому базовые модели и берут с малым смещением, то есть глубокие деревья.

Покрутите $M$ ниже. Слева — $M$ отдельных деревьев (бледные синие) и их среднее (красное); справа — теоретическая кривая $\mathrm{Var}(M)$ с её полом.

In [54]:
# @title Крутилка: усреднение моделей уменьшает разброс
def show_averaging(M):
    truth = f_true(X_GRID)

    # one run: M individual deep trees, each on its own fresh noisy dataset
    base = np.array([fit_predict(*make_dataset(30, 0.25, seed=k), 'дерево', 10)
                     for k in range(M)])
    bagged = base.mean(axis=0)

    # estimate the variance of the averaged model over several independent runs
    runs = np.array([
        np.mean([fit_predict(*make_dataset(30, 0.25, seed=1000 * r + k), 'дерево', 10)
                 for k in range(M)], axis=0)
        for r in range(12)
    ])
    var_bagged = np.mean(runs.var(axis=0))

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for p in base:
        ax[0].plot(X_GRID, p, color='steelblue', alpha=0.18)
    ax[0].plot([], [], color='steelblue', alpha=0.7, label=f'{M} отдельных деревьев')  # legend proxy
    ax[0].plot(X_GRID, truth, 'g-', lw=2.5, label='истина')
    ax[0].plot(X_GRID, bagged, 'r-', lw=2.5, label='их среднее (бэггинг)')
    ax[0].set_ylim(-2.5, 2.5); ax[0].set_xlabel('x'); ax[0].legend(loc='upper right')
    ax[0].set_title(f'M = {M}:  измеренный разброс усреднённой модели = {var_bagged:.3f}')

    # theoretical law (illustrative constants v = 1, rho = 0.3)
    Ms = np.arange(1, 51)
    v, rho = 1.0, 0.3
    var_M = rho * v + (1 - rho) / Ms * v
    ax[1].plot(Ms, var_M)
    ax[1].axhline(rho * v, ls='--', color='gray', label='пол  ρ·v')
    ax[1].axvline(M, ls='--', color='red')
    ax[1].set_xlabel('число моделей M'); ax[1].set_ylabel('дисперсия среднего'); ax[1].legend()
    ax[1].set_title('Теория: Var(M) = ρ·v + (1−ρ)/M · v   (пример: v=1, ρ=0.3)')

    plt.tight_layout()
    plt.show()

interact(show_averaging, M=IntSlider(value=1, min=1, max=25, step=1, description='M моделей'));

interactive(children=(IntSlider(value=1, description='M моделей', max=25, min=1), Output()), _dom_classes=('wi…

**Как читать график (это не ошибка кода!):**
- При $M=1$ «среднее» — это просто **одно** глубокое дерево: дикая ломаная, сильно пляшущая. Бледная линия одна, потому что усреднять пока нечего, а красная совпадает с ней.
- С ростом $M$ бледных деревьев становится больше, и их красное среднее **сглаживается и прижимается к истине**. Число в заголовке — измеренный разброс усреднённой модели (по 12 независимым запускам) — на глазах падает.

Это и есть идея **бэггинга** (англ. *bagging*, от *bootstrap aggregating* — «агрегирование по бутстрепу»): берём глубокие низкосмещённые деревья и усредняем, чтобы убить разброс. А чтобы опустить «пол» $\rho v$, деревья делают **непохожими** друг на друга — так получается **случайный лес** (Random Forest). Об этом — в следующем ноутбуке про ансамбли.

<details>
<summary><b>Для любознательных:</b> вывод формулы дисперсии среднего</summary>

<br>

Пусть $a_1,\dots,a_M$ — прогнозы базовых моделей в фиксированной точке, у каждого дисперсия $\mathrm{Var}(a_i)=v$, а попарная ковариация $\mathrm{Cov}(a_i,a_j)=\rho v$ при $i\neq j$ (по определению корреляции $\rho = \mathrm{Cov}(a_i,a_j)/v$). Дисперсия суммы раскрывается как сумма всех дисперсий и всех попарных ковариаций:

$$\mathrm{Var}\Big(\sum_{i=1}^M a_i\Big) = \sum_{i} \mathrm{Var}(a_i) + \sum_{i \neq j}\mathrm{Cov}(a_i,a_j) = M\,v + M(M-1)\,\rho v.$$

Дисперсия среднего — это дисперсия суммы, делённая на $M^2$:

$$\mathrm{Var}\Big(\tfrac1M\sum_i a_i\Big) = \frac{M\,v + M(M-1)\,\rho v}{M^2} = \frac{v}{M} + \frac{M-1}{M}\,\rho v = \rho v + \frac{1-\rho}{M}\,v.$$

Именно поэтому при $M\to\infty$ остаётся только «пол» $\rho v$, а при $\rho=0$ (независимые модели) разброс падает до нуля как $v/M$.

</details>

## 7. Выводы

Мы проследили мысль целиком:

1. Есть **истинная зависимость** $f(x)$ и **шум** $\varepsilon$. Шум непредсказуем и даёт неустранимый пол ошибки $\sigma^2$.
2. **Сложность модели** — её гибкость. Даже линейная модель имеет (низкую, фиксированную) сложность.
3. Из-за случайности обучающей выборки модель «пляшет». Отсюда два типа ошибки: **смещение** (в среднем мимо) и **разброс** (неустойчивость).
4. Полная ошибка = смещение² + разброс + шум. Сложность снижает смещение, но повышает разброс — **компромисс**, U-образная кривая полной ошибки.
5. Простые модели (линейная) — большое смещение, малый разброс (**недообучение**). Сложные (глубокое дерево) — малое смещение, большой разброс (**переобучение**).
6. **Усреднение** многих разнообразных моделей убивает разброс, не трогая смещение → мост к ансамблям.

**Памятка:**

| | Смещение (bias) | Разброс (variance) |
|---|---|---|
| Что это | в среднем мимо истины | неустойчивость к смене выборки |
| Простая модель (линейная) | большое | малый |
| Сложная модель (глубокое дерево) | малое | большой |
| Симптом | недообучение (underfitting) | переобучение (overfitting) |
| Как уменьшить | усложнить модель | упростить модель / **усреднить** модели / собрать больше данных |

---

## Зачем это всё на практике (даже если вы просто кормите таблицы в CatBoost)

Справедливый вопрос: на работе вы будете читать SQL, собирать таблицы и отдавать их в CatBoost/LightGBM. Где здесь bias и variance? — **Везде.** Понимание BVT — это разница между «тыкать гиперпараметры наугад» и «знать, в какую сторону крутить».

- **Диагноз по двум числам.** Модель отлично на обучении и плохо на валидации? Это **разброс** (переобучение) — лечится упрощением (меньше `depth`, больше регуляризации, меньше итераций / ниже `learning_rate`), бо́льшим объёмом данных, усилением случайности. Плохо и там, и там? Это **смещение** (недообучение) — лечится усложнением модели и новыми признаками. Этот диагноз (разрыв train–valid) вы будете ставить каждый день.
- **Тюнинг — это навигация по кривой компромисса.** Каждый гиперпараметр (`depth`, `l2_leaf_reg`, `learning_rate`, `subsample`, число деревьев) двигает вас по оси bias↔variance. Зная это, перебор параметров превращается из лотереи в осмысленное движение.
- **Выбор модели = выбор, какую ошибку бить.** Случайный лес борется с разбросом, бустинг (CatBoost) — со смещением. Понимая, чего не хватает задаче, вы выбираете инструмент, а не гадаете.
- **Когда «больше данных» поможет, а когда нет.** Данные сбивают разброс, но не смещение. Если упёрлись в смещение — собирать ещё миллион строк бесполезно, надо менять признаки/модель. Это экономит недели работы и деньги компании.
- **Почему нельзя просто взять «самую мощную модель».** Переобученная модель блестит на исторических данных и теряет деньги на новых — ровно та ошибка, за которую в проде платят больно.

Короче: вы будете не «крутить ручки», а понимать, *что* именно крутите и *зачем*. Это и есть то, за что платят senior-инженерам, а не джунам.